# PathSnap: Notebook 01 — Exploratory Data Analysis & Data Quality Audit

This notebook audits the PathSnap raw dataset: file completeness, GPS sampling intervals, geographic extents, and road network topology across training and student-test partitions.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathsnap.io import load_trip, list_trips, resolve_data_dir
from pathsnap.geo import get_trip_crs
from pathsnap.graph import RoadNetwork
from pathsnap.visualize import plot_timestamp_gap_distribution

data_dir = resolve_data_dir()
print(f"Dataset root: {data_dir}")

## 1. Dataset Inventory

In [ ]:
train_trips = list_trips('train', base_dir=data_dir)
test_trips = list_trips('student_test', base_dir=data_dir)
print(f"Total Training Trips: {len(train_trips)}")
print(f"Total Student-Testing Trips: {len(test_trips)}")

# Load inventory summary CSV
inv_df = pd.read_csv('../outputs/csv/dataset_inventory.csv')
inv_df.groupby('partition')[['num_observations', 'num_nodes', 'num_arcs', 'num_route_arcs']].sum()

## 2. GPS Quality & Sampling Regularity

In [ ]:
gps_df = pd.read_csv('../outputs/csv/gps_quality_audit.csv')
exact_1s = gps_df['exact_1s_gaps'].sum()
total_pts = gps_df['num_observations'].sum()
print(f"Exact 1.0s sampling gaps: {exact_1s:,} ({exact_1s/total_pts*100:.2f}%)")
print(f"Gaps > 10s: {gps_df['gaps_gt_10s'].sum():,}")
print(f"Gaps > 60s: {gps_df['gaps_gt_60s'].sum():,}")
print(f"Max observed time gap: {gps_df['max_time_gap'].max():.1f}s")

## 3. Road Network Topology & Independence

In [ ]:
net_df = pd.read_csv('../outputs/csv/network_quality_audit.csv')
print(f"Mean Weakly Connected Components: {net_df['num_wcc'].mean():.1f}")
print(f"Mean Largest WCC Fraction: {net_df['largest_wcc_fraction'].mean()*100:.2f}%")
print(f"Mean Consecutive Route Connectivity: {net_df['route_exact_connection_ratio'].mean()*100:.2f}%")